In [2]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Ensure project root is importable in notebook
workspace_root = Path(os.getcwd()).resolve().parent
if str(workspace_root) not in sys.path:
    sys.path.append(str(workspace_root))

# Load env from project root so LLMProvider can read GROQ/OLLAMA settings
load_dotenv(workspace_root / ".env")

from backend.src.services.pdf_service import extract_text_from_pdf
from backend.src.prompts.build_prompts import BuildPrompts
from backend.src.services.ai_agent.providers import LLMProvider

pdf_path = workspace_root / "pdfs" / "0001_Nguyễn_Văn_An_ClassicBlue.pdf"
cv_text = extract_text_from_pdf(str(pdf_path))

print("[1] Extracted text length:", len(cv_text))
print(cv_text[:500], "...\n")

prompts = BuildPrompts()
provider = LLMProvider()
print("[2] Using provider:", provider.provider.value)

# Build parsing prompt and call provider.generate
parsing_prompt = prompts.build_cv_parsing_prompt(cv_text)
print("[3] Parsing prompt length:", len(parsing_prompt))

try:
    parsing_resp = provider.generate(
        prompt=parsing_prompt,
        system_prompt="You are a precise CV parser. Return JSON only.",
    )
    print("\n[4] LLM result for parsing_prompt (pretty JSON):\n")
    parsing_json = json.loads(parsing_resp.text)
    print(json.dumps(parsing_json, indent=2, ensure_ascii=False))
except Exception as e:
    print("[4] Parsing call failed:", str(e))

# Build section retrieval prompt and call provider.generate
cv_section_items = [
    {
        "cvId": "cv-001",
        "cvName": "Nguyen Van An",
        "sections": {
            "experience": cv_text[:1500],
            "skills": "Python, SQL, Machine Learning, FastAPI",
            "projects": "Recommendation system and CV parsing automation",
        },
    },
    {
        "cvId": "cv-002",
        "cvName": "Tran Thi B",
        "sections": {
            "experience": "Sales and marketing background",
            "skills": "CRM, communication",
            "projects": "Campaign reporting",
        },
    },
]

section_prompt = prompts.build_cv_section_match_prompt(
    question="Tim CV co kinh nghiem machine learning va recommendation system",
    cv_section_items=cv_section_items,
    allowed_sections=["experience", "skills", "projects"],
)
print("\n[5] Section match prompt length:", len(section_prompt))

try:
    section_resp = provider.generate(
        prompt=section_prompt,
        system_prompt="You are a CV retrieval assistant. Return JSON only.",
    )
    print("\n[6] LLM result for section_prompt (pretty JSON):\n")
    section_json = json.loads(section_resp.text)
    print(json.dumps(section_json, indent=2, ensure_ascii=False))
except Exception as e:
    print("[6] Section retrieval call failed:", str(e))

# Optional: quick JSON validation preview
for label, raw_text in [
    ("parsing", locals().get("parsing_resp").text if "parsing_resp" in locals() else None),
    ("section", locals().get("section_resp").text if "section_resp" in locals() else None),
]:
    if not raw_text:
        continue
    try:
        parsed = json.loads(raw_text)
        print(f"\n[7] {label} JSON parsed successfully. Top-level keys:", list(parsed.keys()))
    except Exception as e:
        print(f"\n[7] {label} JSON parse warning:", str(e))

[1] Extracted text length: 1563
Nguyễn Văn An
Senior Software Engineer at FPT Software
+84 90 1234 567 | an.nguyen@example.com | Hà Nội
TÓM TẮT
Software engineer with 5 years of experience in backend development and system architecture, passionate about scalable
solutions and clean code.
KINH NGHIỆM LÀM VIỆC
Vị trí gần nhất
Senior Software Engineer at FPT Software
Số năm KN
5 years
Chi tiết
2016 - 2018: Junior Software Engineer at VNG Corporation, developed mobile gaming modules using
Unity 3D. 2018 - 2021: Software Engineer a ...

[2] Using provider: shopaikey
[3] Parsing prompt length: 2435

[4] LLM result for parsing_prompt (pretty JSON):

{
  "name": "Nguyễn Văn An",
  "phone": "+84 90 1234 567",
  "email": "an.nguyen@example.com",
  "location": "Hà Nội",
  "contact": "+84 90 1234 567",
  "current_job_title": "Senior Software Engineer",
  "educated": true,
  "ever_studied_abroad": false,
  "major": "Computer Science",
  "cpa": null,
  "education": "Bachelor of Science in Computer S

In [8]:
# Cell 2: Test batch scoring prompt with section weights
import importlib
import re
import backend.src.prompts.build_prompts as build_prompts_module
from backend.src.services.ai_agent.providers import LLMProvider

# Reload module to ensure latest BuildPrompts signature is used in notebook kernel
importlib.reload(build_prompts_module)
BuildPrompts = build_prompts_module.BuildPrompts
prompts = BuildPrompts()

# Use higher max_tokens for scoring output to reduce truncation risk
provider_scoring = LLMProvider(max_tokens=4096)
print("[S0] Scoring provider:", provider_scoring.provider.value, "| max_tokens=4096")

candidates = [
    {
        "id": "cv-001",
        "full_name": "Nguyen Van An",
        "current_job_title": "Senior Software Engineer",
        "education_text": "BS Computer Science, MS Software Engineering",
        "experience_text": cv_text,
        "skills_text": "Python, SQL, Machine Learning, FastAPI, Docker",
        "summary_text": "5 years backend experience, scalable systems, CI/CD",
    },
    {
        "id": "cv-002",
        "full_name": "Tran Thi B",
        "current_job_title": "Marketing Specialist",
        "education_text": "BA Business Administration",
        "experience_text": "3 years in digital marketing and campaign analytics",
        "skills_text": "CRM, communication, Google Ads",
        "summary_text": "Marketing-focused profile",
    },
]

job_description_text = """
We are hiring a backend engineer with Python, FastAPI, and Machine Learning experience.
Must have strong API development skills and practical recommendation system exposure.
"""

section_weights = {
    "skills": 0.4,
    "experience": 0.35,
    "projects": 0.15,
    "education": 0.05,
    "summary": 0.05,
}

scoring_prompt = prompts.build_batch_scoring_prompt(
    job_description_text=job_description_text,
    candidates=candidates,
    section_weights=section_weights,
)

print("[S1] Scoring prompt length:", len(scoring_prompt))
print(scoring_prompt[:800], "...\n")

try:
    scoring_resp = provider_scoring.generate(
        prompt=scoring_prompt,
        system_prompt="You are an objective recruitment scoring system. Return JSON only.",
    )
    raw_scoring_text = scoring_resp.text
    print("[S2] Raw LLM result for scoring_prompt:\n")
    print(raw_scoring_text)

    # Try strict parse first
    try:
        scoring_json = json.loads(raw_scoring_text)
        print("\n[S3] Parsed scoring JSON (pretty):\n")
        print(json.dumps(scoring_json, indent=2, ensure_ascii=False))
    except Exception as parse_err:
        print("\n[S3] Strict JSON parse failed:", str(parse_err))

        # Try recovering first JSON object from raw text
        match = re.search(r"\{[\s\S]*\}", raw_scoring_text)
        if match:
            try:
                recovered = json.loads(match.group(0))
                print("\n[S4] Recovered scoring JSON (pretty):\n")
                print(json.dumps(recovered, indent=2, ensure_ascii=False))
            except Exception as recover_err:
                print("[S4] JSON recovery failed:", str(recover_err))
        else:
            print("[S4] No JSON object pattern found in raw output.")
except Exception as e:
    print("[S2] Scoring call failed:", str(e))

[S0] Scoring provider: shopaikey | max_tokens=4096
[S1] Scoring prompt length: 3117
You are an objective recruitment scoring system. Use sectionWeights when calculating scores. Return valid JSON only with the shape shown in responseFormat.

{"jobDescription": "We are hiring a backend engineer with Python, FastAPI, and Machine Learning experience.\nMust have strong API development skills and practical recommendation system exposure.", "sectionWeights": {"skills": 0.4, "experience": 0.35, "projects": 0.15, "education": 0.05, "summary": 0.05}, "candidates": [{"candidateId": "cv-001", "fullName": "Nguyen Van An", "currentJobTitle": "Senior Software Engineer", "education": "BS Computer Science, MS Software Engineering", "experience": "Nguy\u1ec5n V\u0103n An\nSenior Software Engineer at FPT Software\n+84 90 1234 567 | an.nguyen@example.com | H\u00e0 N\u1ed9i\nT\u00d3M T\u1eaeT ...

[S2] Raw LLM result for scoring_prompt:

{"scores":[{"candidateId":"cv-001","totalScore":85.25,"passedThreshol

In [9]:
# Cell 3: Quick summary of scoring run
print("[R1] raw_scoring_text exists:", "raw_scoring_text" in locals())
if "raw_scoring_text" in locals():
    print("[R2] raw_scoring_text length:", len(raw_scoring_text))

parsed_obj = None
if "scoring_json" in locals():
    parsed_obj = scoring_json
    print("[R3] scoring_json parse: success")
elif "recovered" in locals():
    parsed_obj = recovered
    print("[R3] recovered parse: success")
else:
    print("[R3] parse: failed")

if parsed_obj and isinstance(parsed_obj, dict):
    scores = parsed_obj.get("scores", [])
    print("[R4] scores count:", len(scores))
    if scores:
        first = scores[0]
        print("[R5] first candidateId:", first.get("candidateId"))
        print("[R6] first totalScore:", first.get("totalScore"))
        print("[R7] first passedThreshold:", first.get("passedThreshold"))

[R1] raw_scoring_text exists: True
[R2] raw_scoring_text length: 2219
[R3] scoring_json parse: success
[R4] scores count: 2
[R5] first candidateId: cv-001
[R6] first totalScore: 85.25
[R7] first passedThreshold: True
